# 03 — Scope et closures

## Learning Objectives

By the end of this notebook, you will know how to :

- comprendre la règle **LEGB** (Local, Enclosing, Global, Built-in) ;
- lire une variable of the scope englobante dans une function ;
- usesr `global` et `nonlocal` à bon escient (et savoir les éviter) ;
- écrire une **closure** : une function qui capture des variables d'une scope englobante.

## Prérequis

- all the notions previouses ;
- **functions** (def) et **type hints**.

Pas encore vus :

- exceptions (prochain notebook 02_Syntaxe/06).

## Outline

1. Qu'est-ce qu'une scope ?
2. La règle LEGB
3. Lire une globale
4. Modifier une globale : `global`
5. Functions imbriquées et scope `enclosing`
6. Modifier une enclosing : `nonlocal`
7. Closures
8. Common pitfalls
9. Synthèse
10. Exercises

---


## 1. Qu'est-ce qu'une scope ?

Une **scope** (scope) définit où un nom est visible. Les variables déclarées dans une function sont **locales** : elles disparaissent to the sortie of the function.

In [ ]:
def f() -> None:
    x = 42
    print('dans f, x =', x)

f()

In [ ]:
# Hors de f, x n'exists pas
try:
    print(x)
except NameError as err:
    print('error :', err)

---


## 2. La règle LEGB

Quand Python cherche un nom, il parcourt les scopes dans cet ordre :

1. **L**ocal — la function en cours ;
2. **E**nclosing — les functions englobantes, de l'intérieur vers l'extérieur ;
3. **G**lobal — le module (le file / notebook) ;
4. **B**uilt-in — `print`, `len`, `range`…

Le first trouvé gagne. Si rien ne correspond, `NameError`.

---


## 3. Lire une globale

In [ ]:
TAUX: float = 0.2  # constante de module

In [ ]:
def tva(prix_ht: float) -> float:
    return prix_ht * TAUX  # reading de TAUX depuis la portée module

tva(100.0)

Lire une globale est permis sans mot-clé particulier.

---


## 4. Modifier une globale : `global`

Sans `global`, une affectation dans une function creates une **nouvelle** variable locale, same si un nom identique exists au niveau module.

In [ ]:
counter = 0

In [ ]:
def incrementer_faux() -> None:
    counter = counter + 1  # ⚠️ error : counter est considéré local

try:
    incrementer_faux()
except UnboundLocalError as err:
    print('error :', err)

In [ ]:
def incrementer() -> None:
    global counter
    counter = counter + 1

incrementer()
incrementer()
print(counter)

**Usage recommandé :** éviter `global` sauf dans de petits scripts. Préférer **returnsr** la nouvelle valeur.

---


## 5. Functions imbriquées et scope `enclosing`

We can définir une function **à l'intérieur** d'une other. La function interne voit les variables of the function externe.

In [ ]:
def externe() -> None:
    message = 'coucou'
    def interne() -> None:
        print(message)  # lu dans la portée englobante
    interne()

externe()

---


## 6. Modifier une enclosing : `nonlocal`

Pour **modifiesr** une variable of the function englobante (pas du module), on uses `nonlocal`.

In [ ]:
def compteur_fabrique():
    n = 0
    def incrementer() -> int:
        nonlocal n
        n = n + 1
        return n
    return incrementer

c = compteur_fabrique()
print(c())
print(c())
print(c())

---


## 7. Closures

Une **closure** is ae function qui **capture** des variables de sa scope englobante. L'Example ci-dessus is ae closure : `incrementer` capture `n`.

Other Example : une fabrique de functions de multiplication.

In [ ]:
def multiplicateur(facteur: int):
    def inner(x: int) -> int:
        return x * facteur
    return inner

double = multiplicateur(2)
triple = multiplicateur(3)
print(double(5))
print(triple(5))

Each closure garde **son propre** `facteur`, same après que `multiplicateur` a fini de s'exécuter.

---


## 8. Common pitfalls

### Capturer une variable de loop

In [ ]:
fonctions = []
for i in range(3):
    def f() -> int:
        return i
    fonctions.append(f)

# Surprise : toutes renvoient 2 (la dernière value de i)
[fn() for fn in fonctions]

Pour capturer the value *au moment* of the création, on la passe comme parameter by default :

In [ ]:
fonctions = []
for i in range(3):
    def f(i: int = i) -> int:
        return i
    fonctions.append(f)

[fn() for fn in fonctions]

---


## 9. Synthèse

| Situation | Outil |
|---|---|
| Lire une globale | Rien — accès direct |
| Écrire une globale | `global nom` (à éviter) |
| Lire une enclosing | Rien — accès direct |
| Écrire une enclosing | `nonlocal nom` |
| Function qui capture un contexte | closure |

### Règles

1. **Préférer passer / returnsr** plutôt que `global`.
2. Les closures sont la manière idiomatique de « fabriquer » des functions configurées.
3. LEGB : Local > Enclosing > Global > Built-in.

---


## 10. Exercises

### Exercise 1 — Compteur global *(facile)*

Créer une variable `appels = 0` au niveau module. Écrire une function `increment() -> int` qui incrémente `appels` de 1 et returns sa nouvelle valeur.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
appels: int = 0

def increment() -> int:
    """Incrémente le compteur global d'appels."""
    global appels
    appels = appels + 1
    return appels

print(increment())
print(increment())
print(increment())
```

</details>

### Exercise 2 — Accumulateur local *(facile)*

Écrire `somme_jusqua(n: int) -> int` qui computes `1 + 2 + ... + n`. Utiliser uniquement des variables locales.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
def somme_jusqua(n: int) -> int:
    """Renvoie 1+2+...+n (0 si n <= 0)."""
    total = 0
    for i in range(1, n + 1):
        total = total + i
    return total

print(somme_jusqua(10))
```

</details>

### Exercise 3 — Fabrique de puissance *(moyen)*

Écrire `puissance_n(n: int)` which returns une function prenant `x: float` et renvoyant `x ** n`. Tester avec `becausere = puissance_n(2)` puis `becausere(5.0)`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from typing import Callable

def puissance_n(n: int) -> Callable[[float], float]:
    """Renvoie une function x -> x**n."""
    def inner(x: float) -> float:
        return x ** n
    return inner

becausere = puissance_n(2)
cube = puissance_n(3)
print(becausere(5.0))
print(cube(5.0))
```

</details>

### Exercise 4 — Compteur encapsulé *(moyen)*

Écrire `compteur_since(debut: int)` which returns une function interne `next_value() -> int`. Each appel de `next_value` returns the value courante puis l'incrémente. Utiliser `nonlocal`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from typing import Callable

def compteur_since(debut: int) -> Callable[[], int]:
    """Fabrique un compteur incrémental démarrant à `debut`."""
    n = debut
    def next_value() -> int:
        nonlocal n
        valeur = n
        n = n + 1
        return valeur
    return next_value

c = compteur_since(10)
print(c())
print(c())
print(c())
```

</details>

### Exercise 5 — Filtre paramétré *(moyen)*

Écrire `plus_grand_que(seuil: float)` which returns une function prenant une list de floats et renvoyant ceux qui sont strictement supérieurs à `seuil`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from typing import Callable

def plus_grand_que(seuil: float) -> Callable[[list[float]], list[float]]:
    """Fabrique un filtre : > seuil."""
    def inner(valeurs: list[float]) -> list[float]:
        resultat: list[float] = []
        for v in valeurs:
            if v > seuil:
                resultat.append(v)
        return resultat
    return inner

au_dessus_de_10 = plus_grand_que(10.0)
print(au_dessus_de_10([5.0, 12.0, 9.0, 18.0]))
```

</details>

### Exercise 6 — Mémoïsation simple *(difficile)*

Écrire `memoriser(fn)` which takes une function `fn` de `int -> int` et returns une version qui met en cache les résultats dans un dictionary (closure). Tester with ae function lente (`fibonacci` récursif naïf, par Example). Annoter avec `Callable`.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from typing import Callable

def memoriser(fn: Callable[[int], int]) -> Callable[[int], int]:
    """Renvoie une version mémoïsée de fn."""
    cache: dict[int, int] = {}
    def inner(x: int) -> int:
        if x in cache:
            return cache[x]
        resultat = fn(x)
        cache[x] = resultat
        return resultat
    return inner

def fibo(n: int) -> int:
    if n < 2:
        return n
    return fibo(n - 1) + fibo(n - 2)

fibo = memoriser(fibo)
print(fibo(30))
```

</details>

### Exercise 7 — Registre de callbacks *(difficile)*

Écrire `registre()` which returns deux functions dans un tuple : `addsr(cb)` et `appeler_tous()`. Les deux functions partagent (via closure) une list de callbacks. `appeler_tous` les invoque dans l'ordre. Les callbacks prennent et returnsnt un `int`. Afficher les retours.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la Solution</summary>

```python
from typing import Callable

def registre() -> tuple[Callable[[Callable[[int], int]], None], Callable[[int], list[int]]]:
    """Registre de callbacks int -> int."""
    callbacks: list[Callable[[int], int]] = []

    def addsr(cb: Callable[[int], int]) -> None:
        callbacks.append(cb)

    def appeler_tous(valeur: int) -> list[int]:
        return [cb(valeur) for cb in callbacks]

    return addsr, appeler_tous

addsr, appeler = registre()
addsr(lambda x: x + 1)
addsr(lambda x: x * 2)
print(appeler(10))
```

</details>

---


## Ressources externes

- [Scopes et namespaces — tutoriel](https://docs.python.org/3/tutorial/classs.html#python-scopes-and-namespaces)
- [`nonlocal` — reference](https://docs.python.org/3/reference/simple_stmts.html#the-nonlocal-statement)
- [Closures — wiki Python](https://en.wikipedia.org/wiki/Closure_(computer_programming))

---

## Mini-Examples supplémentaires

### LEGB en action — same nom à several niveaux

In [ ]:
x = 'module'

def externe() -> None:
    x = 'enclosing'
    def interne() -> None:
        x = 'local'
        print('local   :', x)
    interne()
    print('enclosing:', x)

externe()
print('module  :', x)

### Une closure capture **la variable**, pas sa valeur

In [ ]:
def fabrique():
    n = 0
    def lire() -> int:
        return n
    def ecrire(v: int) -> None:
        nonlocal n
        n = v
    return lire, ecrire

lire, ecrire = fabrique()
print(lire())
ecrire(42)
print(lire())

### Function-outil pour checksr qu'un nom est visible

In [ ]:
def exists(name: str) -> bool:
    return name in globals() or name in dir(__builtins__)

print(exists('print'))
print(exists('yopyop'))

### Function récursive et scope

In [ ]:
def fact(n: int) -> int:
    if n <= 1:
        return 1
    return n * fact(n - 1)

fact(6)

La function `fact` se référence elle-same : lookup in the scope `module`.

### Pattern : cache caché dans une closure

In [ ]:
from typing import Callable

def avec_cache(fn: Callable[[int], int]) -> Callable[[int], int]:
    cache: dict[int, int] = {}
    def wrap(x: int) -> int:
        if x not in cache:
            cache[x] = fn(x)
        return cache[x]
    return wrap

square = avec_cache(lambda x: x * x)
print(square(5))
print(square(5))  # lu depuis le cache

---

## Quiz flash — checksz vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre comprehension avant de passer au notebook following. Les réponses sont in the bloc `<details>` en dessous.


**Question 1.** Que veut dire `LEGB` ?

<details>
<summary>📖 Réponse</summary>

Local, Enclosing, Global, Built-in — l'ordre in thequel Python résout un nom.

</details>

**Question 2.** Quelle différence between `global` et `nonlocal` ?

<details>
<summary>📖 Réponse</summary>

`global` fait référence au niveau module ; `nonlocal` à une function englobante.

</details>

**Question 3.** Qu'est-ce qu'une closure ?

<details>
<summary>📖 Réponse</summary>

Une function qui **capture** des variables de sa scope englobante.

</details>

**Question 4.** Pourquoi `global` est-il déconseillé ?

<details>
<summary>📖 Réponse</summary>

Parce qu'il creates un couplage invisible : lire un code qui modifies des globales est difficile. Préférer passer et returnsr.

</details>

**Question 5.** Que fait `nonlocal` si la variable n'exists pas in the scope englobante ?

<details>
<summary>📖 Réponse</summary>

Python lève `SyntaxError` à l'analyse — c'is ae erreur de compilation, pas d'exécution.

</details>